# Rover on Rough Terrain — Planner Challenge (C++ edition, 30 min)

A point rover must drive from `start` to `goal` across rough 2D terrain.

**You have:**
- `cost_map(w)` — the `(H,W)` per-cell cost grid you plan under (the objective,
  given to you). Cells `>= ROCK_PENALTY/2` are **untraversable** (obstacles);
  elsewhere, lower is cheaper. `x = column`, `y = row`. Lower path-integral = better.

**Your one task:** write `plan(w, cost_map)` (in `planner_p2.hpp`) → a near-optimal
`(x,y)` path from `start` to `goal` under that cost. The metric integrates cost
along the whole path; the bar is a comfortable tolerance above optimal.
**Grid A* / Dijkstra (8-connected) is the natural choice** and guarantees
near-optimality. (Sampling works too but needs a good init.) You only
`%%writefile planner_p2.hpp`.

### Setup

In [ ]:
import os, subprocess, struct
import numpy as np
if not os.path.exists('field_p2.npy'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field_p2.npy').astype(np.float32)        # (H, W, 3)
meta  = np.load('meta_p2.npz')
H, W = field.shape[:2]

with open('world_p2.bin', 'wb') as f:
    f.write(struct.pack('<ii', H, W))
    f.write(np.ascontiguousarray(field, np.float32).tobytes())
    f.write(np.ascontiguousarray(meta['true_cost'], np.float64).tobytes())
    f.write(np.asarray(meta['start'], np.float64).tobytes())
    f.write(np.asarray(meta['goal'],  np.float64).tobytes())
    f.write(struct.pack('<ddd', float(meta['optimal_cost']), float(meta['tol']),
                        float(meta['rock_sentinel'])))

import sys
sys.path.insert(0, os.getcwd())
from viz_p2 import show_cpp, build_and_run

print('g++ version:',
      subprocess.run(['g++', '--version'], capture_output=True, text=True).stdout.splitlines()[0])
print(f'packed world_p2.bin: field {field.shape} '
      f'| start {meta["start"]} goal {meta["goal"]}')
_chk = subprocess.run(['g++', '-O2', '-std=c++17', 'main_show_p2.cpp', '-o', '_chk_p2'],
                      capture_output=True, text=True)
print('toolchain + shipped sources:', _chk.stderr.strip() or 'OK')

### API (all prebuilt — do not edit; ships in the cloned repo)

Everything is passed the `World w`:
- `w.H`, `w.W`; `w.start`, `w.goal` (`Vec2{x,y}`, `x=col, y=row`).

**Cost maps** are `std::vector<double>`, length `H*W`, row-major: `cost_map[r*W + c]`.
Cells `>= ROCK_PENALTY/2` are untraversable (obstacles).

**Call these:** `cost_map(w)` (the objective you plan under),
`path_cost(w, p, cost_map)` (score a path — use it inside your planner),
`evaluate(w, p, cost_map)` (official PASS/FAIL).

You implement ONLY `plan(const World& w, const std::vector<double>& cost_map)`
in `planner_p2.hpp`.

### The cost map you plan under

In [ ]:
build_and_run('main_show_p2.cpp', [('initial_p2.bin', 'the cost map you plan under')])

## TODO &mdash; `plan` in `planner_p2.hpp` (see the cell)

In [ ]:
%%writefile planner_p2.hpp
// ---- TODO: plan a near-optimal start->goal path under cost_map -----------
// Return an (x,y) Path from w.start to w.goal that minimizes
// path_cost(w, path, cost_map). Cells >= ROCK_PENALTY/2 are untraversable.
// Recommended: 8-connected grid A*/Dijkstra (move cost = neighbor cell cost *
// step length 1 or sqrt(2)) -- guarantees near-optimality, no init issues.
// (Sampling works too but needs a good init; from a straight line it gets stuck.)
#pragma once
#include "costmap_p2.hpp"
#include "metric_p2.hpp"
#include "terrain_p2.hpp"
#include <vector>

inline Path plan(const World& w, const std::vector<double>& cost_map) {
    // TODO: replace this straight line (which ignores cost_map and FAILs).
    Path p(60);
    for (int i = 0; i < 60; ++i) {
        double a = double(i) / 59.0;
        p[i] = { w.start.x + (w.goal.x - w.start.x) * a,
                 w.start.y + (w.goal.y - w.start.y) * a };
    }
    return p;
}

### Run your planner &mdash; goal: `PASS`

In [ ]:
build_and_run('main_run_p2.cpp', [('run_p2.bin', 'planned path over the cost map')])

### Discussion (optional — no code needed)

Be ready to talk through: is your path optimal / complete, and what guarantees
your method gives; A* vs. Dijkstra vs. sampling (RRT*) vs. trajectory-opt; your
heuristic and whether it's admissible; 8- vs 4-connectivity and grid-resolution
vs. cost tradeoffs; `float` vs `double` / cache locality in C++; a safety margin
around rocks; replanning at 50 Hz.